**Imports and paths**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from torchvision import transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

IMAGE_ROOT = (
    PROJECT_ROOT
    / "03_Data_Processed"
    / "mammography_512"
)

MANIFEST_FILE = (
    IMAGE_ROOT
    / "breast_level_image_manifest.csv"
)

IMAGE_CHECKPOINT = (
    PROJECT_ROOT
    / "04_Models"
    / "saved_models"
    / "best_two_view_convnext_tiny.pth"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "multimodal"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "04_Models"
    / "saved_models"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Manifest:", MANIFEST_FILE.exists())
print("Image checkpoint:", IMAGE_CHECKPOINT.exists())

Mounted at /content/drive
Manifest: True
Image checkpoint: True


**Reproducibility**

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


**Load data and rebuild Colab image paths**

In [4]:
df = pd.read_csv(MANIFEST_FILE)

def rebuild_paths(row):

    folder = (
        IMAGE_ROOT
        / row["split"]
        / row["patient_breast_id"]
    )

    return pd.Series({
        "view1_path": str(folder / "view_1.png"),
        "view2_path": str(folder / "view_2.png")
    })

paths = df.apply(
    rebuild_paths,
    axis=1
)

df = pd.concat(
    [df, paths],
    axis=1
)

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "Missing View 1:",
    (~df["view1_path"].apply(lambda x: Path(x).exists())).sum()
)

print(
    "Missing View 2:",
    (~df["view2_path"].apply(lambda x: Path(x).exists())).sum()
)

Train: 1444
Validation: 309
Test: 317
Missing View 1: 0
Missing View 2: 0


**Image transforms**

In [5]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomRotation(5),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

**Clinical feature sets**

In [6]:
FEATURE_SETS = {

    "Fusion_A_without_BIRADS": {
        "numeric": [
            "age"
        ],

        "categorical": [
            "breast_density",
            "breast_side"
        ]
    },

    "Fusion_B_with_BIRADS": {
        "numeric": [
            "age",
            "bi_rads"
        ],

        "categorical": [
            "breast_density",
            "breast_side"
        ]
    }
}

**Clinical preprocessing**

In [7]:
def build_clinical_preprocessor(
    numeric_features,
    categorical_features
):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

    return preprocessor

**Dataset class**

In [8]:
class MultimodalDataset(Dataset):

    def __init__(
        self,
        dataframe,
        clinical_array,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.clinical_array = np.asarray(
            clinical_array,
            dtype=np.float32
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        view1 = Image.open(
            row["view1_path"]
        ).convert("L")

        view2 = Image.open(
            row["view2_path"]
        ).convert("L")

        if self.transform:

            view1 = self.transform(
                view1
            )

            view2 = self.transform(
                view2
            )

        clinical = torch.tensor(
            self.clinical_array[idx],
            dtype=torch.float32
        )

        label = torch.tensor(
            int(row["target"]),
            dtype=torch.long
        )

        return (
            view1,
            view2,
            clinical,
            label,
            row["patient_breast_id"]
        )

**Define the original ConvNeXt image model**

In [9]:
class TwoViewConvNeXt(nn.Module):

    def __init__(self):

        super().__init__()

        weights = ConvNeXt_Tiny_Weights.DEFAULT

        backbone = convnext_tiny(
            weights=weights
        )

        feature_dim = (
            backbone.classifier[2].in_features
        )

        backbone.classifier[2] = nn.Identity()

        self.backbone = backbone

        self.classifier = nn.Sequential(
            nn.Linear(
                feature_dim * 2,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(
                512,
                2
            )
        )


    def forward(
        self,
        view1,
        view2
    ):

        f1 = self.backbone(view1)
        f2 = self.backbone(view2)

        combined = torch.cat(
            [f1, f2],
            dim=1
        )

        return self.classifier(
            combined
        )

**Multimodal fusion model**

In [10]:
class MultimodalFusionModel(nn.Module):

    def __init__(
        self,
        clinical_dim
    ):

        super().__init__()

        # -----------------------------
        # Load trained image model
        # -----------------------------

        image_model = TwoViewConvNeXt()

        image_model.load_state_dict(
            torch.load(
                IMAGE_CHECKPOINT,
                map_location="cpu"
            )
        )

        self.image_backbone = (
            image_model.backbone
        )

        # ConvNeXt-Tiny gives 768 features
        # per image.
        self.image_feature_dim = 768 * 2


        # -----------------------------
        # Freeze image backbone
        # -----------------------------

        for parameter in (
            self.image_backbone.parameters()
        ):

            parameter.requires_grad = False


        # -----------------------------
        # Clinical branch
        # -----------------------------

        self.clinical_branch = (
            nn.Sequential(

                nn.Linear(
                    clinical_dim,
                    32
                ),

                nn.ReLU(),

                nn.Dropout(0.2),

                nn.Linear(
                    32,
                    32
                ),

                nn.ReLU()
            )
        )


        # -----------------------------
        # Fusion classifier
        # -----------------------------

        fusion_dim = (
            self.image_feature_dim
            + 32
        )

        self.fusion_classifier = (
            nn.Sequential(

                nn.Linear(
                    fusion_dim,
                    256
                ),

                nn.ReLU(),

                nn.Dropout(0.4),

                nn.Linear(
                    256,
                    64
                ),

                nn.ReLU(),

                nn.Dropout(0.3),

                nn.Linear(
                    64,
                    2
                )
            )
        )


    def forward(
        self,
        view1,
        view2,
        clinical
    ):

        # Frozen image feature extraction
        with torch.no_grad():

            image_feature1 = (
                self.image_backbone(
                    view1
                )
            )

            image_feature2 = (
                self.image_backbone(
                    view2
                )
            )

        image_features = torch.cat(
            [
                image_feature1,
                image_feature2
            ],
            dim=1
        )

        clinical_features = (
            self.clinical_branch(
                clinical
            )
        )

        fused = torch.cat(
            [
                image_features,
                clinical_features
            ],
            dim=1
        )

        return self.fusion_classifier(
            fused
        )

**Evaluation function**

In [11]:
def evaluate_multimodal(
    model,
    loader,
    criterion
):

    model.eval()

    labels_all = []
    predictions_all = []
    probabilities_all = []
    ids_all = []

    running_loss = 0

    with torch.no_grad():

        for (
            view1,
            view2,
            clinical,
            labels,
            breast_ids
        ) in loader:

            view1 = view1.to(device)
            view2 = view2.to(device)
            clinical = clinical.to(device)
            labels = labels.to(device)

            outputs = model(
                view1,
                view2,
                clinical
            )

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item()
                * labels.size(0)
            )

            probabilities = torch.softmax(
                outputs,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

            predictions_all.extend(
                predictions.cpu().numpy()
            )

            probabilities_all.extend(
                probabilities.cpu().numpy()
            )

            ids_all.extend(
                breast_ids
            )


    y_true = np.array(
        labels_all
    )

    y_pred = np.array(
        predictions_all
    )

    y_prob = np.array(
        probabilities_all
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    metrics = {

        "loss":
            running_loss
            / len(loader.dataset),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "specificity":
            specificity,

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                y_prob
            )
    }

    predictions_df = pd.DataFrame({

        "patient_breast_id":
            ids_all,

        "target":
            y_true,

        "prediction":
            y_pred,

        "malignant_probability":
            y_prob
    })

    return metrics, predictions_df

**Training function**

In [12]:
def train_fusion_model(
    feature_set_name,
    feature_info
):

    print("\n" + "=" * 75)
    print(feature_set_name)
    print("=" * 75)


    numeric_features = (
        feature_info["numeric"]
    )

    categorical_features = (
        feature_info["categorical"]
    )

    all_features = (
        numeric_features
        + categorical_features
    )


    # ---------------------------------
    # Fit preprocessing on TRAIN only
    # ---------------------------------

    preprocessor = (
        build_clinical_preprocessor(
            numeric_features,
            categorical_features
        )
    )

    train_clinical = (
        preprocessor.fit_transform(
            train_df[
                all_features
            ]
        )
    )

    val_clinical = (
        preprocessor.transform(
            val_df[
                all_features
            ]
        )
    )

    test_clinical = (
        preprocessor.transform(
            test_df[
                all_features
            ]
        )
    )

    clinical_dim = (
        train_clinical.shape[1]
    )

    print(
        "Clinical feature dimension:",
        clinical_dim
    )


    # ---------------------------------
    # Save preprocessing object
    # ---------------------------------

    joblib.dump(
        preprocessor,
        MODEL_DIR
        / f"{feature_set_name}_clinical_preprocessor.joblib"
    )


    # ---------------------------------
    # Datasets
    # ---------------------------------

    train_dataset = MultimodalDataset(
        train_df,
        train_clinical,
        transform=train_transform
    )

    val_dataset = MultimodalDataset(
        val_df,
        val_clinical,
        transform=eval_transform
    )

    test_dataset = MultimodalDataset(
        test_df,
        test_clinical,
        transform=eval_transform
    )


    # ---------------------------------
    # Loaders
    # ---------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    # ---------------------------------
    # Model
    # ---------------------------------

    model = MultimodalFusionModel(
        clinical_dim
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=1e-4,
        weight_decay=1e-4
    )


    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2
        )
    )


    # ---------------------------------
    # Training
    # ---------------------------------

    EPOCHS = 15
    PATIENCE = 4

    best_auc = -1
    patience_counter = 0

    history = []

    checkpoint_file = (
        MODEL_DIR
        / f"best_{feature_set_name}.pth"
    )


    for epoch in range(
        1,
        EPOCHS + 1
    ):

        model.train()

        running_loss = 0

        for (
            view1,
            view2,
            clinical,
            labels,
            _
        ) in train_loader:

            view1 = view1.to(device)
            view2 = view2.to(device)
            clinical = clinical.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(
                view1,
                view2,
                clinical
            )

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
                * labels.size(0)
            )


        train_loss = (
            running_loss
            / len(train_loader.dataset)
        )


        val_metrics, _ = (
            evaluate_multimodal(
                model,
                val_loader,
                criterion
            )
        )


        scheduler.step(
            val_metrics["roc_auc"]
        )


        history.append({

            "epoch": epoch,

            "train_loss":
                train_loss,

            "val_loss":
                val_metrics["loss"],

            "val_accuracy":
                val_metrics["accuracy"],

            "val_precision":
                val_metrics["precision"],

            "val_recall":
                val_metrics["recall"],

            "val_specificity":
                val_metrics["specificity"],

            "val_f1":
                val_metrics["f1"],

            "val_roc_auc":
                val_metrics["roc_auc"],

            "val_pr_auc":
                val_metrics["pr_auc"]
        })


        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val AUC: {val_metrics['roc_auc']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )


        if (
            val_metrics["roc_auc"]
            > best_auc
        ):

            best_auc = (
                val_metrics[
                    "roc_auc"
                ]
            )

            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_file
            )

            print(
                "  -> Best model saved."
            )

        else:

            patience_counter += 1


        if (
            patience_counter
            >= PATIENCE
        ):

            print(
                "\nEarly stopping triggered."
            )

            break


    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        RESULTS_DIR
        / f"{feature_set_name}_training_history.csv",
        index=False
    )


    return {
        "feature_set":
            feature_set_name,

        "best_val_auc":
            best_auc,

        "checkpoint":
            checkpoint_file,

        "clinical_dim":
            clinical_dim,

        "preprocessor":
            preprocessor,

        "test_loader":
            test_loader
    }

**Train Fusion A**

In [13]:
fusion_a_result = train_fusion_model(
    "Fusion_A_without_BIRADS",
    FEATURE_SETS[
        "Fusion_A_without_BIRADS"
    ]
)


Fusion_A_without_BIRADS
Clinical feature dimension: 7
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 220MB/s] 


Epoch 01 | Train Loss: 0.1484 | Val Loss: 0.8451 | Val AUC: 0.8640 | Val F1: 0.7857
  -> Best model saved.
Epoch 02 | Train Loss: 0.0567 | Val Loss: 0.9458 | Val AUC: 0.8616 | Val F1: 0.7842
Epoch 03 | Train Loss: 0.0637 | Val Loss: 0.8922 | Val AUC: 0.8626 | Val F1: 0.8105
Epoch 04 | Train Loss: 0.0596 | Val Loss: 0.9497 | Val AUC: 0.8632 | Val F1: 0.8035
Epoch 05 | Train Loss: 0.0450 | Val Loss: 0.9999 | Val AUC: 0.8628 | Val F1: 0.8059

Early stopping triggered.


**Train Fusion B**

In [14]:
fusion_b_result = train_fusion_model(
    "Fusion_B_with_BIRADS",
    FEATURE_SETS[
        "Fusion_B_with_BIRADS"
    ]
)


Fusion_B_with_BIRADS
Clinical feature dimension: 8
Epoch 01 | Train Loss: 0.1600 | Val Loss: 0.8283 | Val AUC: 0.8624 | Val F1: 0.7881
  -> Best model saved.
Epoch 02 | Train Loss: 0.0614 | Val Loss: 0.9001 | Val AUC: 0.8630 | Val F1: 0.8174
  -> Best model saved.
Epoch 03 | Train Loss: 0.0487 | Val Loss: 0.9908 | Val AUC: 0.8638 | Val F1: 0.8000
  -> Best model saved.
Epoch 04 | Train Loss: 0.0558 | Val Loss: 0.9875 | Val AUC: 0.8629 | Val F1: 0.8047
Epoch 05 | Train Loss: 0.0519 | Val Loss: 1.0081 | Val AUC: 0.8646 | Val F1: 0.8000
  -> Best model saved.
Epoch 06 | Train Loss: 0.0444 | Val Loss: 1.0788 | Val AUC: 0.8655 | Val F1: 0.7904
  -> Best model saved.
Epoch 07 | Train Loss: 0.0546 | Val Loss: 0.9880 | Val AUC: 0.8665 | Val F1: 0.8163
  -> Best model saved.
Epoch 08 | Train Loss: 0.0412 | Val Loss: 1.0724 | Val AUC: 0.8678 | Val F1: 0.7879
  -> Best model saved.
Epoch 09 | Train Loss: 0.0286 | Val Loss: 1.1588 | Val AUC: 0.8692 | Val F1: 0.7988
  -> Best model saved.
Epoch 10

**COMPARE MULTIMODAL VALIDATION RESULTS**

In [15]:
# ============================================================
# COMPARE MULTIMODAL VALIDATION RESULTS
# ============================================================

print("=" * 70)
print("MULTIMODAL VALIDATION COMPARISON")
print("=" * 70)

print(
    f"Fusion A without BI-RADS best Val ROC-AUC: "
    f"{fusion_a_result['best_val_auc']:.4f}"
)

print(
    f"Fusion B with BI-RADS best Val ROC-AUC: "
    f"{fusion_b_result['best_val_auc']:.4f}"
)

if (
    fusion_b_result["best_val_auc"]
    >
    fusion_a_result["best_val_auc"]
):

    selected_fusion = fusion_b_result
    selected_name = "Fusion_B_with_BIRADS"

else:

    selected_fusion = fusion_a_result
    selected_name = "Fusion_A_without_BIRADS"


print("\nSelected multimodal model:")
print(selected_name)

print(
    "Best validation ROC-AUC:",
    round(
        selected_fusion["best_val_auc"],
        4
    )
)

MULTIMODAL VALIDATION COMPARISON
Fusion A without BI-RADS best Val ROC-AUC: 0.8640
Fusion B with BI-RADS best Val ROC-AUC: 0.8894

Selected multimodal model:
Fusion_B_with_BIRADS
Best validation ROC-AUC: 0.8894


**Comparison**

In [16]:
multimodal_validation_comparison = pd.DataFrame([
    {
        "model": "Fusion A without BI-RADS",
        "best_validation_roc_auc":
            fusion_a_result["best_val_auc"]
    },
    {
        "model": "Fusion B with BI-RADS",
        "best_validation_roc_auc":
            fusion_b_result["best_val_auc"]
    }
])

multimodal_validation_comparison.to_csv(
    RESULTS_DIR
    / "multimodal_validation_comparison.csv",
    index=False
)

multimodal_validation_comparison

,model,best_validation_roc_auc
0,Fusion A without BI-RADS,0.864009
1,Fusion B with BI-RADS,0.889446


In [17]:
# ============================================================
# FINAL MULTIMODAL TEST EVALUATION
# Selected model: Fusion_B_with_BIRADS
# ============================================================

selected_name = "Fusion_B_with_BIRADS"

selected_checkpoint = (
    MODEL_DIR
    / "best_Fusion_B_with_BIRADS.pth"
)

print("Selected checkpoint:")
print(selected_checkpoint)

# ------------------------------------------------------------
# 1. Rebuild clinical preprocessing for Fusion B
# ------------------------------------------------------------

feature_info = FEATURE_SETS[
    "Fusion_B_with_BIRADS"
]

numeric_features = (
    feature_info["numeric"]
)

categorical_features = (
    feature_info["categorical"]
)

all_features = (
    numeric_features
    + categorical_features
)

# IMPORTANT:
# Fit on TRAIN only, exactly as during model development.
preprocessor = build_clinical_preprocessor(
    numeric_features,
    categorical_features
)

train_clinical = preprocessor.fit_transform(
    train_df[
        all_features
    ]
)

test_clinical = preprocessor.transform(
    test_df[
        all_features
    ]
)

clinical_dim = (
    train_clinical.shape[1]
)

print(
    "Clinical feature dimension:",
    clinical_dim
)

# ------------------------------------------------------------
# 2. Rebuild test dataset / loader
# ------------------------------------------------------------

final_test_dataset = MultimodalDataset(
    test_df,
    test_clinical,
    transform=eval_transform
)

final_test_loader = DataLoader(
    final_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 3. Rebuild selected multimodal architecture
# ------------------------------------------------------------

final_multimodal_model = (
    MultimodalFusionModel(
        clinical_dim
    )
    .to(device)
)

# ------------------------------------------------------------
# 4. Load best checkpoint
# ------------------------------------------------------------

final_multimodal_model.load_state_dict(
    torch.load(
        selected_checkpoint,
        map_location=device
    )
)

final_multimodal_model.eval()

print("\nBest multimodal checkpoint loaded successfully.")

# ------------------------------------------------------------
# 5. Evaluate untouched test set
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

test_metrics, test_predictions = (
    evaluate_multimodal(
        final_multimodal_model,
        final_test_loader,
        criterion
    )
)

# ------------------------------------------------------------
# 6. Print final metrics
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL MULTIMODAL TEST RESULTS")
print("=" * 75)

for key, value in test_metrics.items():

    print(
        f"{key}: {value:.4f}"
    )

# ------------------------------------------------------------
# 7. Save final multimodal metrics
# ------------------------------------------------------------

test_metrics_df = pd.DataFrame([
    {
        "model":
            "Fusion_B_with_BIRADS",

        "architecture":
            "Frozen two-view ConvNeXt-Tiny + clinical MLP",

        "selection_metric":
            "validation_roc_auc",

        "best_validation_roc_auc":
            fusion_b_result[
                "best_val_auc"
            ],

        **test_metrics
    }
])

test_metrics_df.to_csv(
    RESULTS_DIR
    / "final_multimodal_test_metrics.csv",
    index=False
)

# ------------------------------------------------------------
# 8. Add identifying clinical information
# ------------------------------------------------------------

test_predictions = (
    test_predictions
    .merge(
        test_df[
            [
                "patient_breast_id",
                "patient_id",
                "breast_side",
                "classification",
                "split"
            ]
        ],
        on="patient_breast_id",
        how="left",
        validate="one_to_one"
    )
)

test_predictions.to_csv(
    RESULTS_DIR
    / "final_multimodal_test_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Print save confirmation
# ------------------------------------------------------------

print("\nSaved:")
print("- final_multimodal_test_metrics.csv")
print("- final_multimodal_test_predictions.csv")

Selected checkpoint:
/content/drive/MyDrive/Dissertation/04_Models/saved_models/best_Fusion_B_with_BIRADS.pth
Clinical feature dimension: 8

Best multimodal checkpoint loaded successfully.

FINAL MULTIMODAL TEST RESULTS
loss: 0.9923
accuracy: 0.7981
precision: 0.8519
recall: 0.7753
specificity: 0.8273
f1: 0.8118
roc_auc: 0.8717
pr_auc: 0.8914

Saved:
- final_multimodal_test_metrics.csv
- final_multimodal_test_predictions.csv
